# India SMS ROC-AUC: where 0.79 and 0.68 come from

Both numbers score the **same deployed model** (`models/spam_lr.joblib`) on the same out-of-distribution set
(`CloveAI/india-spam-sms`, label 1 = spam). They differ only in how near-identical messages are removed first.

| Variant | Dedup rule | Interval |
|---|---|---|
| A | exact raw text | bootstrap over **templates** |
| B | text after `normalize()` (digits → `num`, URLs, etc.) | bootstrap over **messages**, and over **templates** |

Run from the repo root or from `notebooks/`. Needs `pandas`, `scikit-learn`, `pyarrow`.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from sentinel.classifier import SpamClassifier
from sentinel.preprocess import normalize
import re

# Same template key as src/generalization_report.py: digits->0, URLs->tag, first 6 words.
def template_key(t):
    t = re.sub(r"\d+", "0", t.lower())
    t = re.sub(r"https?://\S+|\S+\.(com|in|net|org)\S*", "<url>", t)
    return " ".join(t.split()[:6])

clf = SpamClassifier.load(ROOT / "models" / "spam_lr.joblib")
print("threshold:", clf.threshold)
print("C:", clf.pipeline[-1].C if hasattr(clf.pipeline[-1], "C") else clf.pipeline.steps[-1][1].C)

## 1. Settings and validation split (what the model was tuned on)

In [ ]:
val = pd.read_csv(ROOT / "data/processed/val.csv")
print(f"val n = {len(val)}, spam = {int(val.label.sum())}, legitimate share = {1 - val.label.mean():.1%}  (all-ham baseline accuracy)")

## 2. Score every India message once

In [ ]:
india = pd.read_parquet(ROOT / "data/ood/india_spam.parquet").reset_index(drop=True)
india["p"] = clf.proba(india.text.tolist())
india["norm"] = india.text.map(normalize)
india["tmpl"] = india.text.map(template_key)
print("raw rows:", len(india), " spam:", int(india.label.sum()))

## 3. Variant A: drop exact-text duplicates

In [ ]:
A = india.drop_duplicates("text").reset_index(drop=True)
print(f"A: n={len(A)}, spam={int(A.label.sum())}, ROC-AUC={roc_auc_score(A.label, A.p):.4f}")

## 4. Variant B: drop duplicates after `normalize()`

In [ ]:
B = india.drop_duplicates("norm").reset_index(drop=True)
print(f"B: n={len(B)}, spam={int(B.label.sum())}, ROC-AUC={roc_auc_score(B.label, B.p):.4f}")

## 5. Intervals
Message bootstrap treats each row as independent. Template bootstrap resamples whole template groups, which is
the honest choice when rows are slot-filled copies of each other (this is why variant A's interval is much wider than its message-level one).

In [ ]:
rng = np.random.default_rng(0)

def boot_messages(df, n=1000):
    y, p = df.label.to_numpy(), df.p.to_numpy(); out = []
    for _ in range(n):
        i = rng.integers(0, len(df), len(df))
        if 0 < y[i].sum() < len(i): out.append(roc_auc_score(y[i], p[i]))
    return np.percentile(out, [2.5, 97.5])

def boot_templates(df, n=1000):
    """Cluster bootstrap as in generalization_report.py: resample templates within each class."""
    by = {k: g.index.to_numpy() for k, g in df.groupby("tmpl")}
    pool = {lab: df[df.label == lab].tmpl.unique() for lab in (0, 1)}; out = []
    for _ in range(n):
        idx = np.concatenate([by[t] for lab in (0, 1) for t in rng.choice(pool[lab], len(pool[lab]))])
        s = df.loc[idx]; out.append(roc_auc_score(s.label, s.p))
    return np.percentile(out, [2.5, 97.5])

rows = []
for name, df in [("A exact-dedup", A), ("B normalize-dedup", B)]:
    rows.append({"variant": name, "n": len(df), "spam": int(df.label.sum()), "spam templates": df[df.label == 1].tmpl.nunique(),
                 "ROC-AUC": round(roc_auc_score(df.label, df.p), 4),
                 "95% CI (messages)": tuple(np.round(boot_messages(df), 3)),
                 "95% CI (templates)": tuple(np.round(boot_templates(df), 3))})
pd.DataFrame(rows)

## 6. Why they differ: how much of the spam is near-duplicate

In [ ]:
sp = india[india.label == 1]
print("spam rows (raw):", len(sp))
print("unique raw texts:", sp.text.nunique(), "| unique after normalize():", sp.norm.nunique(), "| unique template keys (first-6-words):", sp.tmpl.nunique())
print("\nlargest spam templates:")
print(sp.tmpl.value_counts().head(10).to_string())

## Reading the result
- A (exact dedup) is the more flattering figure because slot-filled copies still count many times.
- B is the stricter headline. Report it with its definition.
- Both are optimistic: this set was used while developing the model (the `num` token collapse raised India AUC from 0.39 to 0.79), so neither is a clean out-of-distribution estimate.